# Stage 3 Parsed Demo Validation

Checks parse status, audit output, silver tables, and whether grenade rows look event-level or trajectory/tick-level.

In [ ]:
from pathlib import Path
import pandas as pd

BASE = Path('..')
parse_manifest_path = BASE / 'data/bronze/parse_manifest/parse_manifest.parquet'
parse_audit_path = BASE / 'data/bronze/parse_audit/parse_audit.parquet'
parse_quality_path = BASE / 'data/bronze/parse_quality/parse_quality.parquet'
feature_eligible_path = BASE / 'data/silver/parsed_demos/feature_eligible_demos.parquet'
silver_dir = BASE / 'data/silver/parsed_demos'


In [ ]:
parse_manifest = pd.read_parquet(parse_manifest_path)
parse_manifest['parse_status'].value_counts(dropna=False)

In [ ]:
parse_manifest.groupby(['map_name', 'parse_status']).agg(
    demos=('dem_file_name', 'count'),
    rounds=('rows_rounds', 'sum'),
    ticks=('rows_ticks', 'sum'),
).reset_index()

In [ ]:
parsed = parse_manifest[parse_manifest['parse_status'].eq('parsed')].copy()
parsed['round_count_flag'] = parsed['rows_rounds'].apply(lambda value: 'check' if value < 12 or value > 30 else 'ok')
parsed[['dem_file_name', 'map_name', 'rows_rounds', 'rows_ticks', 'rows_smokes', 'rows_grenades', 'round_count_flag']]

In [ ]:
parse_quality = pd.read_parquet(parse_quality_path)
parse_quality['quality_status'].value_counts(dropna=False)

In [ ]:
parse_quality[parse_quality['quality_status'].eq('suspicious_short_demo')][
    ['dem_file_name', 'inferred_map_name', 'rows_rounds', 'rows_ticks', 'quality_notes']
]

In [ ]:
feature_eligible = pd.read_parquet(feature_eligible_path)
print('feature eligible demos:', len(feature_eligible))
feature_eligible[['dem_file_name', 'inferred_map_name', 'rows_rounds', 'rows_ticks']]

In [ ]:
parsed_total_rounds = parse_quality.loc[parse_quality['parse_status'].eq('parsed'), 'rows_rounds'].sum()
eligible_total_rounds = feature_eligible['rows_rounds'].sum()
pd.DataFrame([
    {'scope': 'parsed_demos', 'demos': int(parse_quality['parse_status'].eq('parsed').sum()), 'rounds': int(parsed_total_rounds)},
    {'scope': 'feature_eligible', 'demos': len(feature_eligible), 'rounds': int(eligible_total_rounds)},
])

In [ ]:
parse_audit = pd.read_parquet(parse_audit_path)
parse_audit[['table_name', 'row_count', 'column_count', 'has_series_id', 'has_map_name', 'has_target_team', 'has_opponent', 'has_tick', 'has_X', 'has_Y', 'has_Z']]

In [ ]:
rounds = pd.read_parquet(silver_dir / 'rounds.parquet')
print('rounds', rounds.shape)
rounds.head()

In [ ]:
tick_cols = ['tick', 'X', 'Y', 'Z', 'health', 'inventory', 'series_id', 'target_team', 'opponent', 'map_name']
ticks = pd.read_parquet(silver_dir / 'ticks.parquet', columns=tick_cols)
print('ticks', ticks.shape)
ticks.head()

In [ ]:
smokes = pd.read_parquet(silver_dir / 'smokes.parquet')
print('smokes', smokes.shape)
smokes.head()

In [ ]:
grenades = pd.read_parquet(silver_dir / 'grenades.parquet')
print('grenades', grenades.shape)
grenades.head()

In [ ]:
if {'entity_id', 'tick'}.issubset(grenades.columns):
    per_entity = grenades.groupby('entity_id')['tick'].nunique().describe()
    granularity = 'trajectory/tick-level' if per_entity.get('mean', 0) > 1 else 'event-level'
else:
    per_entity = pd.Series(dtype='float64')
    granularity = 'unknown'

print('grenades granularity:', granularity)
per_entity